### Filter SAM3 video auto annotation by area threshold

In [1]:
from pathlib import Path
import json
import os
from tqdm import tqdm

os.chdir("../../")

AREA_THRESHOLD = 500  # area < this value will be removed

images_dir = Path("datasets/standard_datasets/images")
anno_dir = Path("datasets/standard_datasets/annotations")

video_ids = [d.name for d in images_dir.iterdir() if d.is_dir()]

total_removed = 0
total_before = 0

for video_id in tqdm(video_ids, desc="filtering annotations"):
    annotation_file = anno_dir / f"annotations_{video_id}.json"

    with open(annotation_file, "r") as f:
        coco_data = json.load(f)

    annotations = coco_data["annotations"]
    before = len(annotations)
    total_before += before

    filtered = [ann for ann in annotations if ann["area"] >= AREA_THRESHOLD]
    removed = before - len(filtered)
    total_removed += removed

    for new_id, ann in enumerate(filtered, start=1):
        ann["id"] = new_id

    coco_data["annotations"] = filtered

    with open(annotation_file, "w") as f:
        json.dump(coco_data, f, indent=2)

print(f"Total annotations before: {total_before}")
print(f"Total annotations removed (area < {AREA_THRESHOLD}): {total_removed}")
print(f"Total annotations after: {total_before - total_removed}")


filtering annotations: 100%|██████████| 124/124 [00:13<00:00,  9.16it/s]

Total annotations before: 122066
Total annotations removed (area < 500): 55548
Total annotations after: 66518


###  clean ori json file in image folder 

In [3]:


deleted_count = 0

for video_id in tqdm(video_ids, desc="cleaning json files"):
    folder_video = images_dir / video_id

    for json_file in folder_video.glob("*.json"):
        json_file.unlink()
        deleted_count += 1

print(f"Deleted {deleted_count} json files from image folders")



cleaning json files: 100%|██████████| 124/124 [00:00<00:00, 14288.29it/s]

Deleted 0 json files from image folders


In [13]:
folder_video = images_dir / "0012"

for json_file in folder_video.glob("*.json"):
    json_file.unlink()
    deleted_count += 1

### del small object annotation

In [4]:
total_removed = 0
total_before = 0

for video_id in tqdm(video_ids, desc="removing small object"):
    annotation_file = anno_dir / f"annotations_{video_id}.json"

    with open(annotation_file, "r") as f:
        coco_data = json.load(f)

    # find category_id for "small object"
    cat_name_to_id = {cat["name"]: cat["id"] for cat in coco_data["categories"]}
    small_obj_id = cat_name_to_id.get("small object")
    if small_obj_id is None:
        continue

    annotations = coco_data["annotations"]
    before = len(annotations)
    total_before += before

    filtered = [ann for ann in annotations if ann["category_id"] != small_obj_id]
    removed = before - len(filtered)
    total_removed += removed

    for new_id, ann in enumerate(filtered, start=1):
        ann["id"] = new_id

    coco_data["annotations"] = filtered

    with open(annotation_file, "w") as f:
        json.dump(coco_data, f, indent=2)

print(f"Total annotations before: {total_before}")
print(f"Annotations with 'small object' label removed: {total_removed}")
print(f"Total annotations after: {total_before - total_removed}")


removing small object: 100%|██████████| 124/124 [00:11<00:00, 11.24it/s]

Total annotations before: 66518
Annotations with 'small object' label removed: 24782
Total annotations after: 41736


### Remove _background_ category from coco json

In [11]:
coco_json = Path("datasets/standard_datasets/images/annotations/coco_instance_segmentation.json")
coco_json_new_name = Path("datasets/standard_datasets/annotations/annotations_0006.json")

with open(coco_json, "r") as f:
    data = json.load(f)

before = len(data["categories"])
data["categories"] = [c for c in data["categories"] if c["name"] != "_background_"]
removed = before - len(data["categories"])

with open(coco_json_new_name, "w") as f:
    json.dump(data, f, indent=2)

print(f"Categories before: {before}, removed: {removed}, after: {len(data['categories'])}")

Categories before: 5, removed: 1, after: 4
